# 本地 Vision + LangChain 工具（mock 实现）

与 `19_local_vision_minimal.ipynb` 同一套 LM Studio / OpenAI 兼容端点，本文件**独立运行**，不修改 19 原文。

- 图片：读本地文件 → base64 `data:` URL（逻辑与 19 一致）。
- 工具：用 `@tool` 注册，**函数体为 mock 固定返回**，可日后替换为真实 API。
- 模型需支持 **function / tool calling**；若不报 `tool_calls`，最后会回退为仅打印模型直接回复。

In [ ]:
%pip install -q "langchain-core>=0.3" "langchain-openai>=0.2"

In [ ]:
import base64
import json
from pathlib import Path

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

BASE_URL = "http://127.0.0.1:1234/v1"
API_KEY = "lm-studio"
MODEL = "qwen/qwen3-vl-4b"

IMAGE_NAME = "微信图片_20260504122723_232_510.jpg"


def resolve_local_image(name: str) -> Path:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        for rel in (
            base / "notebooks" / "19_image" / name,
            base / "19_image" / name,
        ):
            if rel.is_file():
                return rel.resolve()
    raise FileNotFoundError(
        f"找不到 {name}。当前 cwd: {here}。请确认文件在 notebooks/19_image/ 下或改为绝对路径。"
    )


IMAGE_PATH = resolve_local_image(IMAGE_NAME)
suffix = IMAGE_PATH.suffix.lower()
mime = (
    "image/jpeg"
    if suffix in {".jpg", ".jpeg"}
    else "image/png"
    if suffix == ".png"
    else "application/octet-stream"
)
b64 = base64.standard_b64encode(IMAGE_PATH.read_bytes()).decode("ascii")
image_url = f"data:{mime};base64,{b64}"


@tool
def get_mock_weather(region: str) -> str:
    """查询某地区的天气概况。当前为占位实现，不真实请求气象服务。"""
    return f"[MOCK 天气] {region}: 晴间多云，10–15°C，南风 2 级"


@tool
def lookup_scene_tags(scene_hint: str) -> str:
    """根据画面文字描述补充结构化标签。当前为占位，不真实做图像检索。"""
    return f"[MOCK 标签] hint={scene_hint!r} -> 高原, 山地, 人文设施"


tools = [get_mock_weather, lookup_scene_tags]
tools_by_name = {t.name: t for t in tools}

llm = ChatOpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    model=MODEL,
    temperature=0,
)
llm_with_tools = llm.bind_tools(tools)


def _fmt_extra(m: BaseMessage) -> str:
    parts: list[str] = []
    ak = getattr(m, "additional_kwargs", None) or {}
    if ak:
        parts.append(f"additional_kwargs: {ak}")
    rm = getattr(m, "response_metadata", None) or {}
    if rm:
        parts.append(f"response_metadata: {rm}")
    return "\n    " + "\n    ".join(parts) if parts else ""


def _trunc_data_url(url: str, head: int = 80) -> str:
    if len(url) <= head:
        return url
    return url[:head] + "..."


def print_message(tag: str, m: BaseMessage) -> None:
    print(f"\n--- [{tag}] {m.__class__.__name__} ---")
    if isinstance(m, HumanMessage):
        c = m.content
        if isinstance(c, str):
            print(c)
            return
        print(f"local_file: {IMAGE_PATH}")
        for block in c or []:
            if not isinstance(block, dict):
                continue
            btype = block.get("type")
            if btype == "text":
                print("text:", block.get("text", ""))
            elif btype == "image_url":
                inner = block.get("image_url") or {}
                u = inner.get("url", "") if isinstance(inner, dict) else str(inner)
                print("image_url:", _trunc_data_url(str(u)))
            else:
                print(f"{btype}:", block)
    elif isinstance(m, AIMessage):
        if m.content:
            print("content:\n", m.content)
        tcs = getattr(m, "tool_calls", None) or []
        if tcs:
            print("tool_calls (判断):")
            for tc in tcs:
                print(
                    f"  - {tc['name']!r} id={tc['id']!r} args={tc.get('args', tc.get('arguments'))}"
                )
        extra = _fmt_extra(m)
        if extra:
            print(extra.strip())
    elif isinstance(m, ToolMessage):
        print(f"tool_call_id: {m.tool_call_id}")
        print(m.content)
    else:
        print(m)


user_text = (
    "先看图，用一句话概括场景；若合适，调用 lookup_scene_tags 传入简短中文描述；"
    "再调用 get_mock_weather，region 填你推测的大致地理区域（如「青藏高原」）。"
    "最后用中文汇总工具返回。"
)
messages: list[BaseMessage] = [
    HumanMessage(
        content=[
            {"type": "text", "text": user_text},
            {"type": "image_url", "image_url": {"url": image_url}},
        ]
    )
]
print_message("用户", messages[0])

MAX_ROUNDS = 5
for round_i in range(MAX_ROUNDS):
    ai: AIMessage = llm_with_tools.invoke(messages)
    print_message(f"模型 第{round_i + 1}轮", ai)
    messages.append(ai)

    tool_calls = getattr(ai, "tool_calls", None) or []
    if not tool_calls:
        if not ai.content:
            print("(本轮无正文且无 tool_calls)")
        break

    for tc in tool_calls:
        name = tc["name"]
        raw = tc.get("args")
        if raw is None:
            raw = tc.get("arguments", {})
        if isinstance(raw, str):
            raw = json.loads(raw) if raw.strip() else {}
        if not isinstance(raw, dict):
            raw = {}
        out = tools_by_name[name].invoke(raw)
        tm = ToolMessage(content=str(out), tool_call_id=tc["id"])
        print_message(f"工具 → {name}", tm)
        messages.append(tm)
else:
    raise RuntimeError(f"超过 MAX_ROUNDS={MAX_ROUNDS}，请检查模型是否陷入重复 tool 调用。")

print("\n=== 结束 ===")
